# **WEEK 1 — Foundations of Time Series Analysis**

>Jerald B. Bongalos | PhD in Data Science | Asian Institute of Management

---

*Reference:*
>Shumway, R. H., & Stoffer, D. S. *Time Series Analysis and Its Applications*
>(Ch 1.1–1.5, 2.1–2.3, 3.1–3.5)

---

**Purpose of this Notebook**

This notebook develops the **foundational assumptions and tools** that make time series analysis meaningful — before any model is fitted.

**The goal is to understand:**

- what a time series is assumed to be as a stochastic object,
- what conditions justify estimation from a single realization,
- what ACF, PACF, and the Wold decomposition represent,
- and how AR(1) and MA(1) serve as building blocks for everything that follows.

**Learning Outcomes**

By the end of this notebook, you should be able to:

1. Distinguish strict stationarity from weak stationarity and explain why the latter suffices
2. Explain ergodicity and why it justifies estimation from one series
3. Distinguish white noise from martingale difference sequences
4. Interpret ACF and PACF as population quantities
5. State the Wold decomposition and its implications
6. Derive the ACF of AR(1) and MA(1) from first principles
7. Identify AR(1) and MA(1) from simulated ACF/PACF patterns
8. Explain near-unit-root behavior and why unit-root testing is required

---

**Notebook Structure**

| Part | Topic | Type |
|------|-------|------|
| 1 | Stationarity, Ergodicity, and Mixing | Theory + Simulation |
| 2 | White Noise vs Martingale Difference | Theory + Simulation |
| 3 | ACF, PACF, and Wold Decomposition | Theory + Simulation |
| 4 | AR(1): Theory, ACF Derivation, Simulation | Theory + Simulation |
| 5 | MA(1): Theory, ACF Derivation, Simulation | Theory + Simulation |
| 6 | Interpreting Simulations: Linking Theory to Finite Samples | Synthesis |
| 7 | Self-Test Questions | Exam Preparation |

In [ ]:
# ============================================================
# SETUP
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.stattools import acf, pacf, adfuller

np.random.seed(123)

# --- Shared plotting helper ---
def plot_ts_acf_pacf(x, title_prefix="", nlags=30):
    plt.figure(figsize=(12, 3.0))
    plt.plot(x, linewidth=1.2, color="#2c3e50")
    plt.title(f"{title_prefix} — Time Series", fontsize=13)
    plt.xlabel("$t$")
    plt.ylabel("$X_t$")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    a = acf(x, nlags=nlags, fft=True)
    p = pacf(x, nlags=nlags, method="ywm")
    n = len(x)
    ci = 1.96 / np.sqrt(n)

    fig, ax = plt.subplots(1, 2, figsize=(12, 3.0))
    ax[0].stem(range(len(a)), a, basefmt=" ", markerfmt="o", linefmt="-")
    ax[0].axhline(ci, color="red", linestyle="--", alpha=0.5, label="$\\pm 1.96/\\sqrt{n}$")
    ax[0].axhline(-ci, color="red", linestyle="--", alpha=0.5)
    ax[0].set_title(f"{title_prefix} — ACF", fontsize=12)
    ax[0].set_xlabel("Lag $k$")
    ax[0].legend(fontsize=8)
    ax[0].grid(True, alpha=0.3)

    ax[1].stem(range(len(p)), p, basefmt=" ", markerfmt="o", linefmt="-")
    ax[1].axhline(ci, color="red", linestyle="--", alpha=0.5, label="$\\pm 1.96/\\sqrt{n}$")
    ax[1].axhline(-ci, color="red", linestyle="--", alpha=0.5)
    ax[1].set_title(f"{title_prefix} — PACF", fontsize=12)
    ax[1].set_xlabel("Lag $k$")
    ax[1].legend(fontsize=8)
    ax[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# --- Shared AR(1) simulator ---
def simulate_ar1(phi, n=500, sigma=1.0, burnin=200):
    eps = np.random.normal(0, sigma, n + burnin)
    x = np.zeros(n + burnin)
    for t in range(1, n + burnin):
        x[t] = phi * x[t-1] + eps[t]
    return x[burnin:]

# --- Shared MA(1) simulator ---
def simulate_ma1(theta, n=400, sigma=1.0):
    eps = np.random.normal(0, sigma, n + 1)
    return eps[1:] + theta * eps[:-1]

print("Setup complete.")

## **PART 1 — Stationarity, Ergodicity, and Mixing**

> **Why this part matters**
>
> Before fitting models or computing diagnostics, we must understand what a time series
> *is assumed to be* as a stochastic object. Most classical tools rely on assumptions
> about stability, dependence, and long-run behavior. This part introduces those
> assumptions and explains what breaks when they fail.

---

### **1.1 Strict Stationarity**

A time series $\{X_t\}$ is **strictly stationary** if the joint distribution of any
finite collection of observations is invariant under time shifts:

$$
(X_{t_1}, X_{t_2}, \ldots, X_{t_k}) \;\overset{d}{=}\; (X_{t_1+h}, X_{t_2+h}, \ldots, X_{t_k+h})
$$

for all $k$, all $t_1, \ldots, t_k$, and all shifts $h$.

**Consequences:**
- All moments (if they exist) are time-invariant
- The full probabilistic behavior of the process does not change over time
- Extremely strong — almost impossible to verify empirically
- Mainly a theoretical concept used to build intuition

---

### **1.2 Weak (Second-Order) Stationarity**

A time series $\{X_t\}$ is **weakly stationary** (or second-order stationary) if:

1. $\mathbb{E}[X_t] = \mu$ &emsp; (constant mean)
2. $\mathrm{Var}(X_t) = \sigma^2$ &emsp; (constant variance)
3. $\mathrm{Cov}(X_t, X_{t+k}) = \gamma(k)$ &emsp; (depends only on lag $k$, not on $t$)

This focuses only on **first and second moments**, not full distributions.

> **Why weak stationarity is the workhorse assumption**
>
> Weak stationarity ensures that:
> - autocovariances and autocorrelations are well-defined
> - ACF and PACF have stable interpretations
> - ARMA representations are meaningful
> - frequency-domain methods are valid
>
> In practice, series are often **made** weakly stationary through transformation,
> detrending, and differencing.

---

### **1.3 Ergodicity**

Ergodicity answers a fundamental question:

> *How can we learn population properties from a single observed time series?*

A process is **ergodic** (in the mean) if time averages converge to population averages:

$$
\frac{1}{T}\sum_{t=1}^T X_t \;\xrightarrow{\text{a.s.}}\; \mathbb{E}[X_t] \quad \text{as } T \to \infty
$$

**Why ergodicity is essential:**
- Without it, sample means may not estimate true means
- Sample ACFs may not reflect population dependence
- Statistical inference becomes unreliable
- Ergodicity justifies computing expectations from a single realization

---

### **1.4 Mixing**

**Mixing** conditions formalize the idea that distant observations become "almost independent."

- Dependence weakens as the lag increases
- Prevents pathological long-memory behavior
- Central Limit Theorems hold under mixing
- ACF estimation is stable
- Required for asymptotic inference

> **Hierarchy:** Strict stationarity $\supset$ Weak stationarity. Ergodicity and mixing are additional conditions on the dependence structure.

---

### **1.5 What Breaks When These Assumptions Fail**

| Assumption Violated | Consequence |
|---|---|
| Nonstationarity in mean | ACF is undefined or misleading; model estimates are spurious |
| Nonstationarity in variance | Heteroskedastic residuals; invalid confidence intervals |
| Non-ergodicity | Sample statistics do not converge to population quantities |
| No mixing / long memory | CLT fails; standard errors are wrong; tests are oversized |


In [ ]:
# ============================================================
# PART 1 SIMULATION: Stationarity — What it looks like
# ============================================================
# Demonstrates:
#   A) Weakly stationary AR(1) — stable mean, variance, ACF
#   B) Nonstationary (random walk) — drifting mean, growing variance
#   C) Heteroskedastic process — time-varying variance
#   D) ADF test to distinguish stationary from nonstationary
# ============================================================

np.random.seed(123)
n = 500

x_stat = simulate_ar1(0.7, n=n)
x_rw = np.cumsum(np.random.normal(0, 1, n))
eps_hetero = np.random.normal(0, 1, n) * np.linspace(0.5, 3.0, n)

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)

axes[0].plot(x_stat, linewidth=1.0, color="#2c3e50")
axes[0].set_title("A) Weakly Stationary AR(1), $\\phi=0.7$", fontsize=12)
axes[0].axhline(0, color="gray", linestyle="--", alpha=0.5)
axes[0].grid(True, alpha=0.3)

axes[1].plot(x_rw, linewidth=1.0, color="#c0392b")
axes[1].set_title("B) Random Walk (Nonstationary)", fontsize=12)
axes[1].grid(True, alpha=0.3)

axes[2].plot(eps_hetero, linewidth=1.0, color="#8e44ad")
axes[2].set_title("C) Heteroskedastic Process (Variance grows with $t$)", fontsize=12)
axes[2].set_xlabel("$t$")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# --- ADF test ---
print("=" * 60)
print("ADF UNIT ROOT TEST (H0: unit root / nonstationary)")
print("=" * 60)
for label, series in [("AR(1) phi=0.7", x_stat), ("Random Walk", x_rw)]:
    stat, pval, lags, nobs, crit, _ = adfuller(series, autolag="AIC")
    verdict = "REJECT H0 -> Stationary" if pval < 0.05 else "FAIL to reject H0 -> Nonstationary"
    print(f"\n{label}:")
    print(f"  ADF statistic = {stat:.4f}")
    print(f"  p-value       = {pval:.4f}")
    print(f"  Result: {verdict}")

# --- Rolling mean/variance ---
fig, axes = plt.subplots(2, 2, figsize=(12, 6))
window = 50

for col, (label, series) in enumerate([("AR(1) phi=0.7", x_stat), ("Random Walk", x_rw)]):
    s = pd.Series(series)
    axes[0, col].plot(s.rolling(window).mean(), linewidth=1.5, color="#2980b9")
    axes[0, col].set_title(f"{label} — Rolling Mean (w={window})", fontsize=11)
    axes[0, col].grid(True, alpha=0.3)

    axes[1, col].plot(s.rolling(window).var(), linewidth=1.5, color="#e67e22")
    axes[1, col].set_title(f"{label} — Rolling Variance (w={window})", fontsize=11)
    axes[1, col].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nObservation:")
print("  AR(1): rolling mean/variance roughly constant -> weak stationarity")
print("  Random walk: both drift over time -> nonstationary")


## **PART 2 — White Noise vs Martingale Difference**

> **Why this part matters**
>
> After removing trend and seasonality, time series analysis asks: *What remains?*
> Most diagnostics check whether residuals behave like "noise." But not all noise is equivalent.
> Understanding the distinction between **white noise** and **martingale difference sequences**
> is critical for interpreting residual diagnostics.

---

### **2.1 White Noise**

A process $\{\varepsilon_t\}$ is **white noise** if:

- $\mathbb{E}[\varepsilon_t] = 0$
- $\mathrm{Var}(\varepsilon_t) = \sigma^2$ &emsp; (constant)
- $\mathrm{Cov}(\varepsilon_t, \varepsilon_{t+k}) = 0$ for all $k \neq 0$

White noise has **no linear autocorrelation**.

> **What white noise does NOT imply:**
> - independence
> - Gaussianity
> - absence of nonlinear dependence (e.g., ARCH/GARCH effects)
>
> A process can be white noise and still display conditional heteroskedasticity.

---

### **2.2 Martingale Difference Sequence (MDS)**

A process $\{\varepsilon_t\}$ is an **MDS** if:

$$
\mathbb{E}[\varepsilon_t \mid \mathcal{F}_{t-1}] = 0
$$

where $\mathcal{F}_{t-1}$ is all information up to time $t-1$.

This means: **the best forecast of $\varepsilon_t$ given the past is zero**.

---

### **2.3 Relationship**

| Property | White Noise | Martingale Difference |
|---|---|---|
| Zero mean | Yes | Yes |
| Uncorrelated | Yes | Yes |
| No conditional predictability | No | Yes |
| $\mathbb{E}[\varepsilon_t \mid \mathcal{F}_{t-1}] = 0$ | Not required | By definition |

$$
\text{MDS} \;\subset\; \text{White Noise}
$$

Every MDS is white noise, but **not every white noise process is an MDS**.

---

### **2.4 Why This Matters in Practice**

The **Ljung-Box test** checks whether autocorrelations are zero — it tests for *whiteness*, not *unpredictability*.

Therefore:
- Residuals may pass whiteness tests and still be conditionally predictable (e.g., ARCH effects)
- "White" residuals do not guarantee model adequacy
- Always ask: *"white in what sense?"*


In [ ]:
# ============================================================
# PART 2 SIMULATION: White Noise vs ARCH (White but Predictable)
# ============================================================
# Key demonstration:
#   - ARCH(1) process is white noise (zero autocorrelation in levels)
#   - but its SQUARED values are autocorrelated
#   - Ljung-Box on levels passes; on squares it fails
# ============================================================
from statsmodels.stats.diagnostic import acorr_ljungbox

np.random.seed(42)
n = 1000

# --- A) iid Gaussian white noise ---
wn_iid = np.random.normal(0, 1, n)

# --- B) ARCH(1): white noise with conditional heteroskedasticity ---
alpha0, alpha1 = 0.2, 0.7
x_arch = np.zeros(n)
sigma2 = np.zeros(n)
sigma2[0] = alpha0 / (1 - alpha1)

for t in range(1, n):
    sigma2[t] = alpha0 + alpha1 * x_arch[t-1]**2
    x_arch[t] = np.random.normal(0, np.sqrt(sigma2[t]))

# --- Plot time series ---
fig, axes = plt.subplots(2, 1, figsize=(12, 5), sharex=True)
axes[0].plot(wn_iid, linewidth=0.8, color="#2c3e50")
axes[0].set_title("A) iid Gaussian White Noise", fontsize=12)
axes[0].grid(True, alpha=0.3)

axes[1].plot(x_arch, linewidth=0.8, color="#c0392b")
axes[1].set_title("B) ARCH(1) — White Noise with Volatility Clustering", fontsize=12)
axes[1].set_xlabel("$t$")
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# --- ACF of levels vs ACF of squares ---
fig, axes = plt.subplots(2, 2, figsize=(12, 6))
nlags = 30
ci = 1.96 / np.sqrt(n)

for col, (label, series) in enumerate([("iid WN", wn_iid), ("ARCH(1)", x_arch)]):
    a_level = acf(series, nlags=nlags, fft=True)
    a_sq = acf(series**2, nlags=nlags, fft=True)

    axes[0, col].stem(range(len(a_level)), a_level, basefmt=" ")
    axes[0, col].axhline(ci, color="red", linestyle="--", alpha=0.5)
    axes[0, col].axhline(-ci, color="red", linestyle="--", alpha=0.5)
    axes[0, col].set_title(f"{label} — ACF of $X_t$ (levels)", fontsize=11)
    axes[0, col].grid(True, alpha=0.3)

    axes[1, col].stem(range(len(a_sq)), a_sq, basefmt=" ")
    axes[1, col].axhline(ci, color="red", linestyle="--", alpha=0.5)
    axes[1, col].axhline(-ci, color="red", linestyle="--", alpha=0.5)
    axes[1, col].set_title(f"{label} — ACF of $X_t^2$ (squares)", fontsize=11)
    axes[1, col].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# --- Ljung-Box ---
print("=" * 65)
print("LJUNG-BOX TEST (H0: no autocorrelation)")
print("=" * 65)
for label, series in [("iid WN", wn_iid), ("ARCH(1)", x_arch)]:
    lb_level = acorr_ljungbox(series, lags=[10, 20], return_df=True)
    lb_sq    = acorr_ljungbox(series**2, lags=[10, 20], return_df=True)
    print(f"\n{label} — Levels:")
    print(lb_level.to_string())
    print(f"\n{label} — Squared:")
    print(lb_sq.to_string())

print("\nKey insight:")
print("  ARCH(1) passes Ljung-Box on levels (it IS white noise)")
print("  but FAILS on squared values (conditional heteroskedasticity)")
print("  -> Whiteness != unpredictability. The distinction matters.")


## **PART 3 — ACF, PACF, and the Wold Decomposition**

> **Why this part matters**
>
> Once weak stationarity is assumed (or enforced), the dependence structure of a time
> series is summarized by the **ACF** and **PACF**. These are not merely plots — they
> are population quantities that justify why ARMA-type models exist at all.

---

### **3.1 Autocorrelation Function (ACF)**

The ACF at lag $k$ is:

$$
\rho(k) = \frac{\gamma(k)}{\gamma(0)} = \frac{\mathrm{Cov}(X_t, X_{t+k})}{\mathrm{Var}(X_t)}
$$

**Interpretation rules:**
- Slow decay: strong persistence
- Rapid decay: weak dependence
- Oscillatory decay: alternating dependence
- Cutoff after lag $q$: moving-average structure

---

### **3.2 Partial Autocorrelation Function (PACF)**

The PACF at lag $k$ measures the correlation between $X_t$ and $X_{t-k}$ **after removing the linear effects of intermediate lags** $(X_{t-1}, \ldots, X_{t-k+1})$.

$$
\alpha(k) = \mathrm{Corr}\!\big(X_t - \hat{X}_t,\; X_{t-k} - \hat{X}_{t-k}\big)
$$

**Interpretation rules:**
- Cutoff after lag $p$: autoregressive order $p$
- Gradual decay: moving-average effects
- Significant spikes indicate **direct** lag dependence

---

### **3.3 ACF and PACF as Population Objects**

Both ACF and PACF are **population moments**, not sample statistics.

- They exist only if second moments exist
- They are time-invariant under weak stationarity
- **Sample ACF/PACF are estimators** — noisy, especially in small samples

---

### **3.4 The Wold Decomposition Theorem**

> Any zero-mean, weakly stationary, purely nondeterministic time series can be written as:
>
> $$X_t = \sum_{j=0}^{\infty} \psi_j \varepsilon_{t-j}$$
>
> where $\psi_0 = 1$, $\sum \psi_j^2 < \infty$, and $\{\varepsilon_t\}$ is white noise.

**What Wold guarantees:**
- All linear dependence can be represented via past shocks
- ARMA models are **finite-parameter approximations** to this infinite MA
- The $\psi$-weights are fundamental to forecasting (Week 3)

---

### **3.5 When the Wold Decomposition Fails**

Wold holds only under **weak stationarity** for the **purely nondeterministic** component.

It fails when:
- The process is nonstationary (unit root, trend)
- The process has a deterministic component (perfect periodicity)
- Second moments do not exist (infinite variance)

> **Exam-safe statement:** "The Wold decomposition justifies ARMA modeling for any weakly stationary, purely nondeterministic process."


In [ ]:
# ============================================================
# PART 3 SIMULATION: ACF, PACF, and Wold Decomposition
# ============================================================

np.random.seed(123)
n = 400

# A) White noise
wn = np.random.normal(0, 1, n)
plot_ts_acf_pacf(wn, title_prefix="A) White Noise (Wold innovations)", nlags=30)

# B) AR(1) phi=0.6
ar1 = simulate_ar1(0.6, n=n)
plot_ts_acf_pacf(ar1, title_prefix="B) AR(1) $\\phi=0.6$ (Wold-compatible)", nlags=30)

# C) MA(1) theta=0.8
ma1 = simulate_ma1(0.8, n=n)
plot_ts_acf_pacf(ma1, title_prefix="C) MA(1) $\\theta=0.8$ (finite Wold)", nlags=30)

# D) Random walk
rw = np.cumsum(np.random.normal(0, 1, n))
plot_ts_acf_pacf(rw, title_prefix="D) Random Walk (Wold FAILS)", nlags=30)

print("Interpretation summary:")
print("  A) White noise: ACF ~ 0 everywhere -> no linear structure")
print("  B) AR(1): ACF decays geometrically; PACF cuts off at lag 1")
print("  C) MA(1): ACF cuts off at lag 1; PACF decays geometrically")
print("  D) Random walk: ACF decays extremely slowly -> Wold does not apply")


## **PART 4 — AR(1): Theory and ACF Derivation**

> **Why this part matters**
>
> The AR(1) process is the simplest nontrivial stochastic time series model.
> Despite its simplicity, it captures the core ideas behind stationarity,
> persistence, autocorrelation decay, and unit roots.

---

### **4.1 The AR(1) Model**

$$
X_t = \phi X_{t-1} + \varepsilon_t, \qquad \varepsilon_t \sim \mathrm{WN}(0, \sigma^2)
$$

---

### **4.2 Stationarity Condition**

Weakly stationary **if and only if** $|\phi| < 1$.

| $\phi$ value | Behavior |
|---|---|
| $\phi = 0$ | White noise |
| $0 < \phi < 1$ | Positive persistence, geometric ACF decay |
| $\phi \to 1^-$ | Near-unit-root: high persistence, slow ACF decay |
| $\phi = 1$ | Random walk (nonstationary) |
| $\phi < 0$ | Oscillatory behavior |
| $|\phi| > 1$ | Explosive (nonstationary) |

---

### **4.3 Derivation of the ACF**

**Variance.** From $X_t = \phi X_{t-1} + \varepsilon_t$, take variance:

$$
\gamma(0) = \phi^2 \gamma(0) + \sigma^2 \quad\Longrightarrow\quad \boxed{\gamma(0) = \frac{\sigma^2}{1 - \phi^2}}
$$

**Autocovariance.** Multiply both sides by $X_{t-k}$ and take expectations:

$$
\gamma(k) = \phi \, \gamma(k-1) \quad\Longrightarrow\quad \gamma(k) = \phi^k \gamma(0)
$$

**ACF.** Normalize:

$$
\boxed{\rho(k) = \frac{\gamma(k)}{\gamma(0)} = \phi^k}
$$

---

### **4.4 PACF Behavior**

- PACF at lag 1: $\alpha(1) = \phi$
- All higher lags: $\alpha(k) \approx 0$ for $k \geq 2$
- This **cutoff property** is the basis for AR order identification.

---

### **4.5 MA($\infty$) Representation (Connection to Wold)**

By repeated substitution:

$$
X_t = \sum_{j=0}^{\infty} \phi^j \varepsilon_{t-j}
$$

The Wold $\psi$-weights for AR(1) are $\psi_j = \phi^j$. Converges because $|\phi| < 1$.


In [ ]:
# ============================================================
# PART 4 SIMULATION: AR(1) — Theory vs finite-sample behavior
# ============================================================

np.random.seed(123)

def theoretical_acf_ar1(phi, nlags=30):
    return phi ** np.arange(nlags + 1)

def plot_ar1_full(phi, n=400, nlags=30, sigma=1.0):
    x = simulate_ar1(phi, n=n, sigma=sigma)
    a = acf(x, nlags=nlags, fft=True)
    p = pacf(x, nlags=nlags, method="ywm")
    ci = 1.96 / np.sqrt(n)

    fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))

    axes[0].plot(x, linewidth=1.0, color="#2c3e50")
    axes[0].set_title(f"AR(1) $\\phi={phi}$ — Time Series", fontsize=11)
    axes[0].set_xlabel("$t$")
    axes[0].grid(True, alpha=0.3)

    axes[1].stem(range(len(a)), a, basefmt=" ", markerfmt="o", linefmt="-")
    if abs(phi) < 1:
        a_theory = theoretical_acf_ar1(phi, nlags=nlags)
        axes[1].plot(a_theory, linewidth=2.5, color="#e74c3c", label="Theory: $\\rho(k)=\\phi^k$")
        axes[1].legend(fontsize=8)
    axes[1].axhline(ci, color="gray", linestyle="--", alpha=0.4)
    axes[1].axhline(-ci, color="gray", linestyle="--", alpha=0.4)
    axes[1].set_title("ACF (sample vs theory)", fontsize=11)
    axes[1].set_xlabel("Lag $k$")
    axes[1].grid(True, alpha=0.3)

    axes[2].stem(range(len(p)), p, basefmt=" ", markerfmt="o", linefmt="-")
    axes[2].axhline(ci, color="gray", linestyle="--", alpha=0.4)
    axes[2].axhline(-ci, color="gray", linestyle="--", alpha=0.4)
    axes[2].set_title("PACF (expect cutoff at lag 1)", fontsize=11)
    axes[2].set_xlabel("Lag $k$")
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    if abs(phi) < 1:
        var_theory = sigma**2 / (1 - phi**2)
        var_sample = np.var(x)
        print(f"  phi={phi}: Var(theory)={var_theory:.3f}, Var(sample)={var_sample:.3f}")

# --- Multiple phi values ---
print("=" * 60)
print("AR(1) DIAGNOSTICS")
print("=" * 60)

for phi in [0.2, 0.6, 0.9, -0.5]:
    plot_ar1_full(phi, n=400, nlags=30)

# --- Near-unit-root vs unit root ---
print("\n" + "=" * 60)
print("NEAR-UNIT-ROOT vs UNIT ROOT")
print("=" * 60)

np.random.seed(123)
x_near = simulate_ar1(0.98, n=400, burnin=300)
x_rw = np.cumsum(np.random.normal(0, 1, 400))

fig, axes = plt.subplots(2, 3, figsize=(15, 7))
nlags = 30
ci = 1.96 / np.sqrt(400)

for row, (label, series) in enumerate([
    ("Near-unit-root: $\\phi=0.98$", x_near),
    ("Random Walk: $\\phi=1.0$", x_rw)
]):
    a = acf(series, nlags=nlags, fft=True)
    p = pacf(series, nlags=nlags, method="ywm")

    axes[row, 0].plot(series, linewidth=1.0)
    axes[row, 0].set_title(f"{label} — Time Series", fontsize=10)
    axes[row, 0].grid(True, alpha=0.3)

    axes[row, 1].stem(range(len(a)), a, basefmt=" ")
    axes[row, 1].axhline(ci, color="red", linestyle="--", alpha=0.4)
    axes[row, 1].axhline(-ci, color="red", linestyle="--", alpha=0.4)
    axes[row, 1].set_title("ACF", fontsize=10)
    axes[row, 1].grid(True, alpha=0.3)

    axes[row, 2].stem(range(len(p)), p, basefmt=" ")
    axes[row, 2].axhline(ci, color="red", linestyle="--", alpha=0.4)
    axes[row, 2].axhline(-ci, color="red", linestyle="--", alpha=0.4)
    axes[row, 2].set_title("PACF", fontsize=10)
    axes[row, 2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\nKey takeaway:")
print("  phi=0.98 (stationary) and phi=1.0 (nonstationary) look nearly identical")
print("  in finite samples. This is WHY unit-root tests are required.")


## **PART 5 — MA(1): Theory and ACF Derivation**

> **Why this part matters**
>
> The MA(1) process is the simplest model where dependence arises **not from persistence**
> but from a **single past shock**. It is the dual of AR(1) and completes the foundation
> needed for ARMA modeling in Week 2.

---

### **5.1 The MA(1) Model**

$$
X_t = \varepsilon_t + \theta \, \varepsilon_{t-1}, \qquad \varepsilon_t \sim \mathrm{WN}(0, \sigma^2)
$$

---

### **5.2 Stationarity**

MA(1) is **always weakly stationary** for any value of $\theta$, because it is a finite linear combination of white noise. No restriction on $\theta$ is needed.

---

### **5.3 Invertibility Condition**

For unique parameter estimation, we require **invertibility**: $|\theta| < 1$.

Without invertibility, $\theta$ and $1/\theta$ produce **the same autocovariance structure**, making the model non-identifiable.

> **Exam-critical:** Causality is for AR; invertibility is for MA. They are not interchangeable.

---

### **5.4 Derivation of the ACF**

**Variance:**

$$
\gamma(0) = \mathrm{Var}(\varepsilon_t + \theta \varepsilon_{t-1}) = \sigma^2(1 + \theta^2)
$$

**Lag 1 autocovariance:**

$$
\gamma(1) = \mathrm{Cov}(\varepsilon_t + \theta \varepsilon_{t-1}, \; \varepsilon_{t-1} + \theta \varepsilon_{t-2}) = \theta \sigma^2
$$

**Lag $k \geq 2$:**

$$
\gamma(k) = 0
$$

**ACF:**

$$
\boxed{\rho(1) = \frac{\theta}{1 + \theta^2}, \qquad \rho(k) = 0 \;\text{ for } k \geq 2}
$$

The ACF **cuts off after lag 1**. This is the signature of MA(1).

> **Note:** The maximum possible value of $|\rho(1)|$ is 0.5, achieved when $\theta = \pm 1$.

---

### **5.5 PACF Behavior**

For MA(1):
- PACF **decays geometrically** (tailing off)
- It does **not** cut off at any finite lag

This is the mirror image of AR(1), where ACF decays and PACF cuts off.

---

### **5.6 AR($\infty$) Representation (Invertibility)**

Under invertibility ($|\theta| < 1$), MA(1) can be written as:

$$
X_t = \sum_{j=1}^{\infty} (-\theta)^j X_{t-j} + \varepsilon_t
$$

This is the dual of AR(1)'s MA($\infty$) representation.

---

### **5.7 Summary: AR(1) vs MA(1) Signatures**

| Property | AR(1) | MA(1) |
|---|---|---|
| ACF | Geometric decay: $\rho(k) = \phi^k$ | Cutoff after lag 1 |
| PACF | Cutoff after lag 1 | Geometric decay |
| Stationarity requires | $|\phi| < 1$ | Always stationary |
| Invertibility requires | Always invertible | $|\theta| < 1$ |
| Wold representation | MA($\infty$): $\psi_j = \phi^j$ | Already MA(1): $\psi_0=1, \psi_1=\theta$ |


In [ ]:
# ============================================================
# PART 5 SIMULATION: MA(1) — Theory vs finite-sample behavior
# ============================================================

np.random.seed(123)

def theoretical_acf_ma1(theta, nlags=30):
    """Theoretical ACF for MA(1)."""
    rho = np.zeros(nlags + 1)
    rho[0] = 1.0
    rho[1] = theta / (1 + theta**2)
    return rho

def plot_ma1_full(theta, n=400, nlags=30, sigma=1.0):
    x = simulate_ma1(theta, n=n, sigma=sigma)
    a = acf(x, nlags=nlags, fft=True)
    p = pacf(x, nlags=nlags, method="ywm")
    ci = 1.96 / np.sqrt(n)

    fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))

    axes[0].plot(x, linewidth=1.0, color="#2c3e50")
    axes[0].set_title(f"MA(1) $\\theta={theta}$ — Time Series", fontsize=11)
    axes[0].set_xlabel("$t$")
    axes[0].grid(True, alpha=0.3)

    rho1 = theta / (1 + theta**2)
    axes[1].stem(range(len(a)), a, basefmt=" ", markerfmt="o", linefmt="-")
    a_theory = theoretical_acf_ma1(theta, nlags=nlags)
    axes[1].plot(a_theory, linewidth=2.5, color="#e74c3c",
                 label=f"Theory: $\\rho(1)={rho1:.3f}$")
    axes[1].legend(fontsize=8)
    axes[1].axhline(ci, color="gray", linestyle="--", alpha=0.4)
    axes[1].axhline(-ci, color="gray", linestyle="--", alpha=0.4)
    axes[1].set_title("ACF (expect cutoff at lag 1)", fontsize=11)
    axes[1].set_xlabel("Lag $k$")
    axes[1].grid(True, alpha=0.3)

    axes[2].stem(range(len(p)), p, basefmt=" ", markerfmt="o", linefmt="-")
    axes[2].axhline(ci, color="gray", linestyle="--", alpha=0.4)
    axes[2].axhline(-ci, color="gray", linestyle="--", alpha=0.4)
    axes[2].set_title("PACF (expect geometric decay)", fontsize=11)
    axes[2].set_xlabel("Lag $k$")
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

# --- Multiple theta values ---
print("=" * 60)
print("MA(1) DIAGNOSTICS")
print("=" * 60)

for theta in [0.3, 0.8, -0.5]:
    plot_ma1_full(theta, n=400, nlags=30)

# --- Invertibility demonstration ---
print("\n" + "=" * 60)
print("INVERTIBILITY: theta vs 1/theta PRODUCE SAME ACF")
print("=" * 60)

np.random.seed(777)
theta1 = 0.5
theta2 = 1 / theta1  # = 2.0 (non-invertible)

x1 = simulate_ma1(theta1, n=2000)
x2 = simulate_ma1(theta2, n=2000)

acf1 = acf(x1, nlags=20, fft=True)
acf2 = acf(x2, nlags=20, fft=True)

fig, ax = plt.subplots(1, 1, figsize=(10, 3.5))
ax.plot(acf1, linewidth=2.5, marker="o", markersize=4, label=f"$\\theta={theta1}$ (invertible)")
ax.plot(acf2, linewidth=2.5, marker="s", markersize=4, linestyle="--",
        label=f"$\\theta={theta2}$ (non-invertible)")
ax.axhline(0, color="gray", linestyle="-", alpha=0.3)
ax.set_title("MA(1) ACF Comparison: $\\theta$ vs $1/\\theta$", fontsize=12)
ax.set_xlabel("Lag $k$")
ax.set_ylabel("ACF")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

rho1_a = theta1 / (1 + theta1**2)
rho1_b = theta2 / (1 + theta2**2)
print(f"\nTheoretical rho(1):")
print(f"  theta = {theta1:.2f}  ->  rho(1) = {rho1_a:.4f}")
print(f"  theta = {theta2:.2f}  ->  rho(1) = {rho1_b:.4f}")
print(f"\n  Same rho(1)! This is why invertibility is required.")
print(f"  Convention: always choose |theta| < 1.")


## **PART 6 — Interpreting Simulations: Linking Theory to Finite Samples**

> **Why this part matters**
>
> Theory tells us how a stochastic process behaves *in expectation*.
> Simulation shows us how that behavior appears in **finite samples**, which is what
> we actually observe in practice.

---

### **6.1 Stationarity Condition in Practice**

As $\phi$ increases toward 1:
- Shocks persist longer
- The series becomes more "trendy"
- Variance appears to grow slowly over time
- The process begins to resemble a random walk

Near-unit-root processes (e.g., $\phi = 0.98$) are **technically stationary** but
behave very similarly to nonstationary series in finite samples.

---

### **6.2 The Identification Table (Exam-Critical)**

| Process | ACF Pattern | PACF Pattern |
|---|---|---|
| White Noise | $\approx 0$ everywhere | $\approx 0$ everywhere |
| AR($p$) | Tails off (geometric/oscillatory decay) | Cuts off after lag $p$ |
| MA($q$) | Cuts off after lag $q$ | Tails off (geometric/oscillatory decay) |
| ARMA($p,q$) | Tails off after lag $q$ | Tails off after lag $p$ |
| Random Walk | Extremely slow decay (spurious) | Spike at lag 1, then $\approx 0$ |

> **Exam-safe statement:** "ACF and PACF guide identification, but they do not uniquely determine the model. They suggest candidates; diagnostics decide."

---

### **6.3 Finite-Sample Realities**

- Sample ACFs and PACFs are noisy estimators
- Small spurious spikes are normal and expected
- Near-unit-root ACFs may appear flat, mimicking nonstationarity
- Order identification is **probabilistic**, not exact
- Unit-root tests (ADF) are required — visual inspection alone is insufficient

---

### **6.4 Connecting to Wold**

- Stationary AR and MA processes admit MA($\infty$) representations (Wold)
- Linear dependence is fully captured by the $\psi$-weights
- ACF and PACF summarize this dependence in interpretable form

When stationarity fails:
- ACF does not decay properly
- Wold decomposition fails
- **Differencing** is required to restore validity (Week 2)


## **PART 7 — Self-Test: Exam Preparation Questions**

> Work through these without looking at notes first. Then verify against the material above.

---

### **Conceptual Questions (Oral Exam Style)**

**Q1.** What is the difference between strict and weak stationarity? Why is weak stationarity sufficient for ARMA modeling?

**Q2.** What does ergodicity guarantee, and why does it matter that we only observe one realization?

**Q3.** A colleague says "my residuals pass the Ljung-Box test, so my model is adequate." What is missing from this argument?

**Q4.** State the Wold decomposition theorem. What does it guarantee about the existence of ARMA representations?

**Q5.** Derive the ACF of AR(1) from the model definition. What assumptions are needed?

**Q6.** Derive the ACF of MA(1). Why does $\rho(k) = 0$ for $k \geq 2$?

**Q7.** Two MA(1) models with $\theta = 0.5$ and $\theta = 2.0$ produce the same autocovariance structure. Why? What resolves this?

**Q8.** You observe a sample ACF that decays very slowly. Is this an AR process with high persistence or a nonstationary process? How would you decide?

---

### **Computation Questions**

**Q9.** For AR(1) with $\phi = 0.8$ and $\sigma^2 = 1$:
- Compute $\gamma(0)$, $\gamma(1)$, $\gamma(5)$
- Compute $\rho(1)$, $\rho(5)$, $\rho(10)$

**Q10.** For MA(1) with $\theta = -0.6$ and $\sigma^2 = 1$:
- Compute $\gamma(0)$, $\gamma(1)$, $\gamma(2)$
- Compute $\rho(1)$

**Q11.** For AR(1) with $\phi = 0.9$: how many lags until $|\rho(k)| < 0.1$?

---

### **Identification Challenge**

**Q12.** Run the cell below. From the ACF and PACF plots alone, identify whether each process is AR(1), MA(1), white noise, or a random walk. Then check your answers.


In [ ]:
# ============================================================
# PART 7: IDENTIFICATION CHALLENGE
# Four mystery processes. Identify each from ACF/PACF patterns.
# ============================================================

np.random.seed(999)
n = 400

processes = {
    "Mystery A": np.random.normal(0, 1, n),
    "Mystery B": simulate_ar1(0.85, n=n),
    "Mystery C": simulate_ma1(-0.7, n=n),
    "Mystery D": np.cumsum(np.random.normal(0, 1, n)),
}

for label, x in processes.items():
    plot_ts_acf_pacf(x, title_prefix=label, nlags=25)

# --- Answers ---
print("\n" * 3)
print("=" * 60)
print("ANSWERS (scroll down after making your guesses)")
print("=" * 60)
print()
print("Mystery A: White Noise")
print("  -> ACF ~ 0 everywhere, PACF ~ 0 everywhere")
print()
print("Mystery B: AR(1) with phi = 0.85")
print("  -> ACF decays geometrically (slowly)")
print("  -> PACF has one significant spike at lag 1")
print()
print("Mystery C: MA(1) with theta = -0.7")
print("  -> ACF has one significant spike at lag 1 (negative), then ~ 0")
print("  -> PACF shows geometric (oscillatory) decay")
print()
print("Mystery D: Random Walk (nonstationary)")
print("  -> ACF decays extremely slowly (spurious persistence)")
print("  -> PACF has dominant spike at lag 1, rest ~ 0")
print("  -> Looks like AR(1) PACF, but the slow ACF is the giveaway")


In [ ]:
# ============================================================
# PART 7: COMPUTATION ANSWERS
# ============================================================
import math

print("=" * 60)
print("Q9: AR(1), phi=0.8, sigma^2=1")
print("=" * 60)
phi, sigma2 = 0.8, 1.0
gamma0 = sigma2 / (1 - phi**2)
print(f"  gamma(0) = sigma^2/(1-phi^2) = {gamma0:.4f}")
print(f"  gamma(1) = phi * gamma(0)    = {phi * gamma0:.4f}")
print(f"  gamma(5) = phi^5 * gamma(0)  = {phi**5 * gamma0:.4f}")
print(f"  rho(1)  = phi^1  = {phi**1:.4f}")
print(f"  rho(5)  = phi^5  = {phi**5:.4f}")
print(f"  rho(10) = phi^10 = {phi**10:.4f}")

print(f"\n{'=' * 60}")
print("Q10: MA(1), theta=-0.6, sigma^2=1")
print("=" * 60)
theta, sigma2 = -0.6, 1.0
gamma0_ma = sigma2 * (1 + theta**2)
gamma1_ma = theta * sigma2
rho1_ma = theta / (1 + theta**2)
print(f"  gamma(0) = sigma^2(1+theta^2) = {gamma0_ma:.4f}")
print(f"  gamma(1) = theta * sigma^2    = {gamma1_ma:.4f}")
print(f"  gamma(2) = 0")
print(f"  rho(1) = theta/(1+theta^2)    = {rho1_ma:.4f}")

print(f"\n{'=' * 60}")
print("Q11: AR(1), phi=0.9 — lags until |rho(k)| < 0.1")
print("=" * 60)
phi = 0.9
k_threshold = math.ceil(math.log(0.1) / math.log(abs(phi)))
print(f"  Need phi^k < 0.1")
print(f"  k > log(0.1)/log(0.9) = {math.log(0.1)/math.log(0.9):.1f}")
print(f"  Answer: k = {k_threshold} lags")
print(f"  Verification: rho({k_threshold}) = {phi**k_threshold:.4f}")
